# Day 17 — Design Pattern: **Factory** (for Agents)

**Phase 1 · Module 2: AWS Bedrock & AgentCore** · ~30 min

Day 14 you used **Strategy** to swap *how* a model is called (Bedrock vs Azure — one interface, interchangeable engines). Today's twin problem: your app has several **kinds** of agent — a support agent, a fraud agent, a loan agent — and something has to decide *which one to build* from a request. Scatter that construction logic (`if agent_type == ...`) across the codebase and every new agent means edits everywhere. The **Factory pattern** centralises creation behind one call: `AgentFactory.create("fraud") → BaseAgent`. Callers ask for a type; the factory owns the wiring.

> **Runnable & keyless.** Agents return canned strings — the factory *machinery* is the lesson, the agent bodies are stubs where real Bedrock/LangGraph agents (this afternoon) would go.

## 1. The smell — construction logic sprayed everywhere

Without a factory, every place that needs an agent grows its own `if/elif` to build one — duplicated, and each new agent type means hunting down all of them. Creation is a *responsibility*; right now nobody owns it.

In [1]:
def handle_request_bad(agent_type, message):
    # construction tangled into the caller, duplicated at every call site
    if agent_type == "support":
        agent = {"role": "support", "model": "claude-haiku"}
    elif agent_type == "fraud":
        agent = {"role": "fraud",   "model": "claude-sonnet"}   # fraud needs a stronger model
    else:
        raise ValueError(agent_type)
    return f"[{agent['role']}/{agent['model']}] handling {message!r}"

print(handle_request_bad("support", "reset my password"))
print(handle_request_bad("fraud",   "unrecognised £5000 charge"))

[support/claude-haiku] handling 'reset my password'
[fraud/claude-sonnet] handling 'unrecognised £5000 charge'


☝ The knowledge that "fraud needs Sonnet" is baked into this caller — and any other caller that builds a fraud agent must repeat it. Add a `loan` agent and you edit every branch like this. Construction wants **one owner** so those decisions live in a single place.

## 2. A common product interface — `BaseAgent`

A factory returns objects the caller treats **uniformly**, so first define the shared contract. `BaseAgent` is an ABC with a `handle(message)` method — every agent, whatever its type, honours it (same idea as Day 14's `LLMProvider`, but here the interface is the *product* a factory makes).

In [2]:
from abc import ABC, abstractmethod

class BaseAgent(ABC):
    model = "claude-haiku"                       # sensible default; subclasses override
    @abstractmethod
    def handle(self, message: str) -> str: ...

    def __repr__(self):
        return f"<{type(self).__name__} model={self.model}>"

☝ The factory's whole value is that callers can hold a `BaseAgent` and call `.handle()` without knowing the concrete class. Defining the interface *first* is what makes the returned objects interchangeable — the factory decides *which*, the interface guarantees *how* it's used.

## 3. Concrete agents — the products

Each agent type subclasses `BaseAgent`, sets its own model, and implements `handle`. These are the "products" the factory will build. Note each encodes its own config (fraud → Sonnet) in **one** place now.

In [3]:
class SupportAgent(BaseAgent):
    model = "claude-haiku"                       # cheap + fast for FAQs
    def handle(self, message):
        return f"[support] resolving: {message!r}"

class FraudAgent(BaseAgent):
    model = "claude-sonnet"                       # stronger model for risk calls
    def handle(self, message):
        return f"[fraud] investigating (model={self.model}): {message!r}"

class LoanAgent(BaseAgent):
    model = "claude-sonnet"
    def handle(self, message):
        return f"[loan] assessing eligibility: {message!r}"

print(FraudAgent(), "->", FraudAgent().handle("£5000 charge in Lagos"))

<FraudAgent model=claude-sonnet> -> [fraud] investigating (model=claude-sonnet): '£5000 charge in Lagos'


☝ "Fraud uses Sonnet" now lives **once**, on `FraudAgent`, not at every call site (contrast §1). Each product is self-contained: its model, its behaviour. The factory won't need to know these details — it just needs to know which class maps to which name.

## 4. The factory — one place that builds by type

`AgentFactory.create(name)` looks the name up in a **registry** and constructs the right product. This is the single owner of "which class, wired how". Callers never see a concrete class — they pass a string and get a `BaseAgent`.

In [4]:
class AgentFactory:
    _registry = {                                # name -> product class
        "support": SupportAgent,
        "fraud":   FraudAgent,
        "loan":    LoanAgent,
    }

    @classmethod
    def create(cls, agent_type: str) -> BaseAgent:
        try:
            return cls._registry[agent_type]()   # build the product
        except KeyError:
            raise ValueError(f"unknown agent {agent_type!r}; have {list(cls._registry)}")

    @classmethod
    def register(cls, name, product):            # extend without editing create()
        cls._registry[name] = product

for t in ("support", "fraud", "loan"):
    agent = AgentFactory.create(t)
    print(agent, "->", agent.handle("hello"))

<SupportAgent model=claude-haiku> -> [support] resolving: 'hello'
<FraudAgent model=claude-sonnet> -> [fraud] investigating (model=claude-sonnet): 'hello'
<LoanAgent model=claude-sonnet> -> [loan] assessing eligibility: 'hello'


☝ `create()` is the one place construction happens; `handle_request` from §1 collapses to `AgentFactory.create(agent_type).handle(message)` — no branching, no duplicated config. The caller depends only on the factory and `BaseAgent`. That's the pattern: **centralise creation, return a common interface**.

## 5. Config-driven + extensible — add an agent with one call

Because the factory reads a registry, ops can pick agents from config, and a *new* agent type is a single `register()` — no edit to `create()` or any caller (Open/Closed, again). A plugin can even register itself on import.

In [5]:
# route by config/data, not hardcoded branches
ROUTING = {"password": "support", "charge": "fraud", "mortgage": "loan"}
def route(message):
    kind = next((a for k, a in ROUTING.items() if k in message.lower()), "support")
    return AgentFactory.create(kind).handle(message)

print(route("unrecognised charge on my card"))
print(route("help with my mortgage application"))

# add a brand-new agent type with ONE line — no factory edits
class ComplaintsAgent(BaseAgent):
    def handle(self, message): return f"[complaints] logging: {message!r}"
AgentFactory.register("complaints", ComplaintsAgent)
print(AgentFactory.create("complaints").handle("I want to file a complaint"))

[fraud] investigating (model=claude-sonnet): 'unrecognised charge on my card'
[loan] assessing eligibility: 'help with my mortgage application'
[complaints] logging: 'I want to file a complaint'


☝ Routing became data (`ROUTING`), and `ComplaintsAgent` joined the system with one `register()` call — `create()`, `route()`, and every caller stayed untouched. The factory turned "add an agent type" from a cross-codebase edit into a one-line registration. That's the scalability win.

## 6. Factory vs Strategy — sibling patterns

Both hide concrete classes behind an interface; they answer **different questions**:

| | **Strategy** (Day 14) | **Factory** (today) |
|---|---|---|
| Question | *how* to do one thing | *which* object to build |
| Varies | the algorithm/engine (Bedrock vs Azure) | the product type (support vs fraud) |
| Swap point | inject a strategy into a context | ask the factory for a type |

They compose: a `FraudAgent` (built by the **Factory**) can call the model through an `LLMProvider` (**Strategy**). This afternoon's Bedrock notebook does exactly that — an agent built for a role, talking to Bedrock through a swappable model interface.

## 7. Recap + checklist

| Piece | Role in Factory |
|---|---|
| `BaseAgent` (ABC) | common **product interface** callers depend on |
| `SupportAgent` / `FraudAgent` / … | concrete **products**, own config in one place |
| `AgentFactory.create(name)` | single **creator** — build by type from a registry |
| `AgentFactory.register(...)` | extend by adding, not editing (Open/Closed) |
| Factory vs Strategy | *which to build* vs *how to run* — they compose |

**Your 30 min today**
- [ ] Define `BaseAgent` with an abstract `handle()`.
- [ ] Write 2–3 concrete agents that set their own model.
- [ ] Build `AgentFactory.create(agent_type)` off a registry; handle unknown types.
- [ ] Add a new agent via `register()` — prove no caller changed.

